# ML-07 — Baseline Action Score and Top-10 Review

This notebook tests two observed signals, encodes one transparent rule, writes a ranked review queue, and reviews its top ten. The rule uses only fields available in the trailing 90-day snapshot; it does not use `trend_direction`, `trend_pct`, or any label-derived field.


## 1. Signal checks and rule reasoning

The rule idea is: **review pages that are both stale and still visible**. Staleness is linked to FlyRank's refresh flags (`days_since_last_update` / `freshness_tier`), while 90-day impressions are a demand/volume signal behind the quick-win idea.

I will check each signal in buckets using observed CTR and position, and print `n` for every bucket. Verdicts are directional checks, not causal claims. The proposed rule is `stale_visible`: `days_since_last_update >= 180` and `impressions_90d >= 500`. Its action label is `refresh_review`; all other rows receive `monitor`. The score is `impressions_90d` for rows satisfying both conditions and `0` otherwise, so the queue prioritizes the largest observed audience within this one reason code.


In [11]:

from pathlib import Path

cwd = Path.cwd()
print("Current directory:", cwd)

# Search the project directory for the exact filename
project_root = cwd.parent.parent

matches = list(
    project_root.rglob("content_refresh_anonymized.csv")
)

if matches:
    for path in matches:
        print("Found:", path)
else:
    print("CSV not found within:", project_root)

from pathlib import Path
import pandas as pd

matches = list(
    Path.cwd().parent.parent.rglob("content_refresh_anonymized.csv")
)

if not matches:
    raise FileNotFoundError(
        "CSV not found. Check the project directory and dataset location."
    )

data_path = matches[0]
print("Loading:", data_path)

df = pd.read_csv(data_path)

if df.empty:
    raise ValueError("Starter data is empty")

print("Dataset shape:", df.shape)
display(df.head())


Current directory: d:\FlyRank-ML-Internship-Starter\work\notebooks
Found: d:\FlyRank-ML-Internship-Starter\data\raw\content_refresh_anonymized.csv
Loading: d:\FlyRank-ML-Internship-Starter\data\raw\content_refresh_anonymized.csv
Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Build the ranked queue (writes the CSV)

One rule only: `stale_visible` means at least 180 days since update and at least 500 impressions in the observed 90-day window. The score is transparent (`impressions_90d` when the rule matches, otherwise zero); the reason code and action label are deterministic.


In [12]:
stale_visible = (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
df["reason_code"] = np.where(stale_visible, "stale_visible", "not_stale_visible")
df["action_label"] = np.where(stale_visible, "refresh_review", "monitor")
df["score"] = np.where(stale_visible, df["impressions_90d"], 0).astype(float)
df["rank"] = df["score"].rank(method="first", ascending=False).astype(int)

queue_columns = [
    "rank", "content_id", "client_id", "score", "reason_code", "action_label",
    "impressions_90d", "days_since_last_update", "avg_position", "ctr", "content_age_days",
]
queue = df[queue_columns].sort_values(["score", "content_id"], ascending=[False, True]).reset_index(drop=True)
output_path = root / "work" / "outputs" / "baseline_action_score.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(output_path, index=False)

receipt = {
    "rows_scored": int(len(queue)),
    "refresh_review_rows": int(stale_visible.sum()),
    "monitor_rows": int((~stale_visible).sum()),
    "top_score": float(queue["score"].max()),
    "rule": "days_since_last_update >= 180 and impressions_90d >= 500",
    "features_used": ["days_since_last_update", "impressions_90d"],
    "label_or_future_fields_used": [],
}
(root / "work" / "outputs" / "baseline_action_score_metrics.json").write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")
print(f"Wrote {len(queue):,} ranked rows to {output_path}")
print(queue.head(10).to_string(index=False))


Wrote 30,000 ranked rows to d:\FlyRank-ML-Internship-Starter\work\notebooks\work\outputs\baseline_action_score.csv
 rank           content_id         client_id   score   reason_code   action_label  impressions_90d  days_since_last_update  avg_position  ctr  content_age_days
    1 content_cf56e2e2e282 client_7f2253d7e2 61678.0 stale_visible refresh_review            61678                     194          19.7 0.15               231
    2 content_7368877ea310 client_7f2253d7e2 59472.0 stale_visible refresh_review            59472                     194          24.8 0.13               231
    3 content_1bfaa38ff26c client_7f2253d7e2 25715.0 stale_visible refresh_review            25715                     194          22.2 0.23               231
    4 content_0a91db491d14 client_7f2253d7e2 13299.0 stale_visible refresh_review            13299                     193          10.5 0.49               231
    5 content_5feee3994adb client_7f2253d7e2  7812.0 stale_visible refresh_review    

## 3. Top-10 review

Each row gets the action, why it is in the queue, and a concrete condition that would make the pick wrong. These are decision-support notes, not claims that refresh will cause improvement.


In [13]:
for _, row in queue.head(10).iterrows():
    why = f"{int(row['impressions_90d']):,} observed impressions and {int(row['days_since_last_update'])} days since update meet stale_visible."
    wrong = "Wrong if the page was intentionally left unchanged, the impressions are not qualified demand, or a recent unpublished update is missing from the snapshot."
    print(f"{int(row['rank']):02d}. {row['content_id']} — action={row['action_label']}; why={why}; what would make it wrong={wrong}")


01. content_cf56e2e2e282 — action=refresh_review; why=61,678 observed impressions and 194 days since update meet stale_visible.; what would make it wrong=Wrong if the page was intentionally left unchanged, the impressions are not qualified demand, or a recent unpublished update is missing from the snapshot.
02. content_7368877ea310 — action=refresh_review; why=59,472 observed impressions and 194 days since update meet stale_visible.; what would make it wrong=Wrong if the page was intentionally left unchanged, the impressions are not qualified demand, or a recent unpublished update is missing from the snapshot.
03. content_1bfaa38ff26c — action=refresh_review; why=25,715 observed impressions and 194 days since update meet stale_visible.; what would make it wrong=Wrong if the page was intentionally left unchanged, the impressions are not qualified demand, or a recent unpublished update is missing from the snapshot.
04. content_0a91db491d14 — action=refresh_review; why=13,299 observed imp

## 4. Weak picks + leakage check

The weakest positive picks are still stale and visible, but have the smallest audience among positive scores. They are weaker because the rule has no position, CTR, quality, or business-priority gate. A reviewer should verify that the impressions are meaningful before acting.


In [14]:
print(queue[queue["action_label"] == "refresh_review"].tail(3).to_string(index=False))
forbidden = {"trend_direction", "trend_pct", "is_declining_label"}
used = {"days_since_last_update", "impressions_90d"}
assert not used & forbidden
assert set(receipt["features_used"]).isdisjoint(forbidden)
assert output_path.exists()
print("Leakage check passed: no future-window or label-derived fields used.")


 rank           content_id         client_id  score   reason_code   action_label  impressions_90d  days_since_last_update  avg_position  ctr  content_age_days
   15 content_72496874f806 client_4ec9599fc2  821.0 stale_visible refresh_review              821                     301           5.8 0.24               301
   16 content_6226ee6adc91 client_d029fa3a95  545.0 stale_visible refresh_review              545                     183          17.8 0.18               232
   17 content_074ba6ead17b client_d029fa3a95  533.0 stale_visible refresh_review              533                     183          48.0 0.00               232
Leakage check passed: no future-window or label-derived fields used.


## Self-check

- [x] Two visible signal tables include `n` and one-word verdicts.
- [x] One rule has a score, one reason code, and an action label.
- [x] The notebook writes `work/outputs/baseline_action_score.csv`.
- [x] Top ten rows include action, rationale, and what would make each wrong.
- [x] No future-window or label-derived inputs are used.
- [x] Metrics receipt is committed; the generated CSV remains ignored.
